# SwingSight Exact Club-Marking CNN Training

This notebook uses the same master images and manifest as the club-head detector. It crops the annotated number, letter, or loft region and trains the exact-marking model that SwingSight uses after a club is classified as an Iron or Wedge.


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch>=2.2.0", "torchvision>=0.17.0", "pandas>=2.0.0", "pillow>=10.0.0",
    "--no-warn-script-location",
])
print("Training packages are ready.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import sys
import pandas as pd
from PIL import Image

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the SwingSight-AI project.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MASTER_DIR = PROJECT_ROOT / "data" / "club_training"
IMAGES_DIR = MASTER_DIR / "images"
MANIFEST_PATH = MASTER_DIR / "annotations" / "club_manifest.csv"
DERIVED_DIR = MASTER_DIR / "derived" / "club_marking"
MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "club_marking_cnn.pt"

print("Master dataset:", MASTER_DIR)
print("Manifest:", MANIFEST_PATH)
print("App output:", MODEL_PATH)


## One shared club-image library

Both club models use the same master images. Keep the original images once, then derive task-specific train/validation folders from one annotation file.

```text
data/club_training/
  images/
    any_folder_you_want/
      image_0001.jpg
  annotations/
    club_manifest.csv
  derived/                    # created by these notebooks
```

The CSV uses one row per image. Required columns:

`image_path,split,five_way_label,head_x,head_y,head_w,head_h,marking_label,mark_x,mark_y,mark_w,mark_h`

- `image_path` is relative to `images/`.
- `split` is `train` or `val`. Keep a source image in one split only.
- `five_way_label` is `driver, wood, hybrid, iron,` or `wedge`.
- `head_*` is the normalized YOLO box for the club head.
- `marking_*` is the normalized box around only the number, letter, or loft. Leave every marking field blank when no readable marking is visible.

This is one dataset, not three copies of the same images.


In [ ]:
CLASS_NAMES = [
    "1", "2", "3", "4", "5", "6", "7", "8", "9",
    "p", "a", "g", "s", "l",
    "50", "52", "54", "56", "58", "60",
]
REQUIRED_COLUMNS = [
    "image_path", "split", "five_way_label",
    "head_x", "head_y", "head_w", "head_h",
    "marking_label", "mark_x", "mark_y", "mark_w", "mark_h",
]

manifest = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
missing = [column for column in REQUIRED_COLUMNS if column not in manifest.columns]
if missing:
    raise ValueError(f"Manifest is missing columns: {missing}")

manifest["split"] = manifest["split"].str.strip().str.lower()
for column in ["mark_x", "mark_y", "mark_w", "mark_h"]:
    manifest[column] = pd.to_numeric(manifest[column], errors="coerce")
manifest["marking_label"] = manifest["marking_label"].str.strip().str.lower()

marked = manifest[manifest["marking_label"].isin(CLASS_NAMES)].copy()
if marked.empty:
    raise ValueError("No valid marking rows found. Add marking_label and marking box values for readable irons/wedges.")
if marked[["mark_x", "mark_y", "mark_w", "mark_h"]].isna().any().any():
    raise ValueError("Each usable marking row needs mark_x, mark_y, mark_w, and mark_h.")
if not ((marked[["mark_x", "mark_y", "mark_w", "mark_h"]] >= 0).all().all()
        and (marked[["mark_x", "mark_y", "mark_w", "mark_h"]] <= 1).all().all()):
    raise ValueError("Marking boxes must be normalized between 0 and 1.")
if not marked["split"].isin(["train", "val"]).all():
    raise ValueError("Every marked row needs a train or val split.")

missing_images = [path for path in marked["image_path"] if not (IMAGES_DIR / path).is_file()]
if missing_images:
    raise FileNotFoundError(f"Missing master images, first few: {missing_images[:10]}")
marked.groupby(["split", "marking_label"]).size().unstack(fill_value=0)


In [ ]:
# Create tight marking crops from the same master images.
# A little padding keeps the number/loft readable without showing the whole club.
for split in ("train", "val"):
    for label in CLASS_NAMES:
        (DERIVED_DIR / split / label).mkdir(parents=True, exist_ok=True)

def crop_normalized(image: Image.Image, x: float, y: float, w: float, h: float, padding: float = 0.08) -> Image.Image:
    left = max(0.0, x - w / 2 - padding * w)
    top = max(0.0, y - h / 2 - padding * h)
    right = min(1.0, x + w / 2 + padding * w)
    bottom = min(1.0, y + h / 2 + padding * h)
    return image.crop((int(left * image.width), int(top * image.height), int(right * image.width), int(bottom * image.height)))

for row in marked.itertuples(index=False):
    source = IMAGES_DIR / row.image_path
    with Image.open(source) as opened:
        crop = crop_normalized(opened.convert("RGB"), row.mark_x, row.mark_y, row.mark_w, row.mark_h)
    destination = DERIVED_DIR / row.split / row.marking_label / f"{source.stem}_{row.marking_label}.jpg"
    crop.save(destination, quality=95)

counts = {
    split: {
        label: len(list((DERIVED_DIR / split / label).glob("*")))
        for label in CLASS_NAMES
    }
    for split in ("train", "val")
}
print("Smallest train classes:", sorted(counts["train"].items(), key=lambda item: item[1])[:5])
print("Smallest validation classes:", sorted(counts["val"].items(), key=lambda item: item[1])[:5])


In [ ]:
if min(counts["train"].values()) < 20 or min(counts["val"].values()) < 5:
    raise ValueError(
        "Each class needs at least 20 train and 5 validation crops. "
        "Use the same master images, but add more readable marking annotations for the small classes."
    )
print("Marking dataset is ready.")


## Train and validate


In [ ]:
import subprocess

command = [
    sys.executable, "scripts/train_club_cnn.py",
    "--task", "club_marking",
    "--data-dir", str(DERIVED_DIR),
    "--output", str(MODEL_PATH),
    "--epochs", "35",
    "--batch-size", "32",
    "--learning-rate", "1e-3",
    "--workers", "0",
]
subprocess.check_call(command, cwd=PROJECT_ROOT)
print("Saved:", MODEL_PATH)


In [ ]:
import torch
from swingsight.club_cnn import classify_image

checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
assert checkpoint["task"] == "club_marking"
assert checkpoint["class_names"] == CLASS_NAMES
print("Validation accuracy:", checkpoint["validation_accuracy"])

sample_path = next(path for path in (DERIVED_DIR / "val").rglob("*") if path.is_file())
with Image.open(sample_path) as opened:
    prediction = classify_image(
        opened.convert("RGB"),
        checkpoint_path=MODEL_PATH,
        expected_task="club_marking",
        minimum_confidence=0.0,
    )
print("Actual:", sample_path.parent.name)
print("Predicted:", prediction.label, "| confidence:", round(prediction.confidence, 3))


Restart SwingSight after training. The app will use the same shared image library-derived model to identify a **7 Iron**, **Pitching Wedge**, **56° Wedge**, and similar exact markings.
